# **Data Processing**

#### **STEP 1: REVIEW DATA**
- import library
- import data
- review data (menampikan 5 baris pertama)
- melihat tipe data
- cek duplikasi dan *missing value*

In [1]:
# library
import pandas as pd

In [2]:
# data healt outcomes daily
hlt01 = pd.read_csv('../[1] Dataset DA Gelar Rasa 2026/HLT-01_health_outcomes_daily.csv')
hlt01.head()

,date,time,hospital_id,dow,temp_c,dptp_c,rhum_pct,resp_admissions_under5,resp_admissions_over60,asthma_emergency_cases,copd_admissions,resp,cvd,death
0,2023-12-31,1.0,RS-XXX-12,Sunday,23.7,21.2,87.6,6.0,5.0,2.0,3.0,21.0,6.0,2.0
1,2023-12-31,1.0,RS-XXX-13,Sunday,24.4,22.1,88.7,4.0,7.0,4.0,4.0,27.0,8.0,1.0
2,2023-12-31,1.0,RS-XXX-16,Sunday,24.5,22.0,87.5,4.0,4.0,3.0,4.0,15.0,5.0,1.0
3,2023-12-31,1.0,RS-XXX-19,Sunday,24.0,21.8,88.9,4.0,3.0,1.0,3.0,12.0,6.0,2.0
4,2023-12-31,1.0,RS-XXX-20,Sunday,24.1,21.8,88.7,6.0,4.0,4.0,3.0,18.0,8.0,0.0


In [3]:
# tipe data
hlt01.dtypes

date                       object
time                      float64
hospital_id                object
dow                        object
temp_c                    float64
dptp_c                    float64
rhum_pct                  float64
resp_admissions_under5    float64
resp_admissions_over60    float64
asthma_emergency_cases    float64
copd_admissions           float64
resp                      float64
cvd                       float64
death                     float64
dtype: object

In [4]:
# missing value per colums
hlt01.isna().sum()

date                      0
time                      0
hospital_id               0
dow                       0
temp_c                    0
dptp_c                    0
rhum_pct                  0
resp_admissions_under5    0
resp_admissions_over60    0
asthma_emergency_cases    0
copd_admissions           0
resp                      0
cvd                       0
death                     0
dtype: int64

In [5]:
# duplicated data
hlt01.duplicated().sum()

np.int64(0)

#### **STEP 2: MAKE PRIMARY KEY**
dalam table ini tidak ada kolom yang memiliki nilai unik (identitas baris), ini penting untuk:
1. Indexing dan kebutuhan join
2. Menjamin integritas data jangka panjang

In [6]:
# primary key colum
nilai_hlt_id = hlt01['hospital_id'].astype(str).str.replace('-', '') + '-' + hlt01['date'].astype(str)
hlt01.insert(0, 'hlt_id', nilai_hlt_id) # insert colom at index 0

hlt01.head()

,hlt_id,date,time,hospital_id,dow,temp_c,dptp_c,rhum_pct,resp_admissions_under5,resp_admissions_over60,asthma_emergency_cases,copd_admissions,resp,cvd,death
0,RSXXX12-2023-12-31,2023-12-31,1.0,RS-XXX-12,Sunday,23.7,21.2,87.6,6.0,5.0,2.0,3.0,21.0,6.0,2.0
1,RSXXX13-2023-12-31,2023-12-31,1.0,RS-XXX-13,Sunday,24.4,22.1,88.7,4.0,7.0,4.0,4.0,27.0,8.0,1.0
2,RSXXX16-2023-12-31,2023-12-31,1.0,RS-XXX-16,Sunday,24.5,22.0,87.5,4.0,4.0,3.0,4.0,15.0,5.0,1.0
3,RSXXX19-2023-12-31,2023-12-31,1.0,RS-XXX-19,Sunday,24.0,21.8,88.9,4.0,3.0,1.0,3.0,12.0,6.0,2.0
4,RSXXX20-2023-12-31,2023-12-31,1.0,RS-XXX-20,Sunday,24.1,21.8,88.7,6.0,4.0,4.0,3.0,18.0,8.0,0.0


In [7]:
# check status primary key (unique)
hlt01['hlt_id'].nunique() == hlt01.shape[0]

True

#### **STEP 3: FIX COLUM** `TIME`

Dalam **Casebook** tertulis keterangan kolom `time` seperti berikut: <br>
"Nomor urut hari sejak awal pengamatan". Artinya hari pertama dalam dataset memiliki nilai 1, hari ke-2 memiliki nilai 2 dan seterusnya<br>

Tetapi, terdapat kesalahan data pada **2024-01-01** sampai dengan **2026-10-14** yang harusnya **n** hari tetapi malah berisi value **n-1** hari. <br>

In [8]:
hlt01.groupby('date')['time'].first().reset_index()

,date,time
0,2023-12-31,1.0
1,2024-01-01,1.0
2,2024-01-02,2.0
3,2024-01-03,3.0
4,2024-01-04,4.0
...,...,...
1014,2026-10-10,1014.0
1015,2026-10-11,1015.0
1016,2026-10-12,1016.0
1017,2026-10-13,1017.0


terlihat diatas, data awal dimulai pada tanggal 2023-12-31 yang harusnya memiliki nilai 1 (sudah benar), tetapi untuk tanggal 2024-01-01 juga memiliki value 1 **(harusnya 2)**. <br>
ini menyebabkan value dari kolom `time` (selain 2023-12-31) salah, dan **ini yang bakal kita perbaiki.**

In [9]:
hlt01['date'] = pd.to_datetime(hlt01['date']) # convert to datetime for compute calculation

# fix value
hlt01['time'] = (hlt01['date'] - hlt01['date'].min()).dt.days + 1
hlt01.head()

,hlt_id,date,time,hospital_id,dow,temp_c,dptp_c,rhum_pct,resp_admissions_under5,resp_admissions_over60,asthma_emergency_cases,copd_admissions,resp,cvd,death
0,RSXXX12-2023-12-31,2023-12-31,1,RS-XXX-12,Sunday,23.7,21.2,87.6,6.0,5.0,2.0,3.0,21.0,6.0,2.0
1,RSXXX13-2023-12-31,2023-12-31,1,RS-XXX-13,Sunday,24.4,22.1,88.7,4.0,7.0,4.0,4.0,27.0,8.0,1.0
2,RSXXX16-2023-12-31,2023-12-31,1,RS-XXX-16,Sunday,24.5,22.0,87.5,4.0,4.0,3.0,4.0,15.0,5.0,1.0
3,RSXXX19-2023-12-31,2023-12-31,1,RS-XXX-19,Sunday,24.0,21.8,88.9,4.0,3.0,1.0,3.0,12.0,6.0,2.0
4,RSXXX20-2023-12-31,2023-12-31,1,RS-XXX-20,Sunday,24.1,21.8,88.7,6.0,4.0,4.0,3.0,18.0,8.0,0.0


In [10]:
# chack output
hlt01.groupby('date')['time'].first().reset_index()

,date,time
0,2023-12-31,1
1,2024-01-01,2
2,2024-01-02,3
3,2024-01-03,4
4,2024-01-04,5
...,...,...
1014,2026-10-10,1015
1015,2026-10-11,1016
1016,2026-10-12,1017
1017,2026-10-13,1018


#### **STEP 4: FIX DATA TYPE**
Beberapa data numerik masih ada yang memiliki tipe data yang kurang tepat, seperti `resp`, `resp_admissions_over60` dan lainnya yang harusnya *integer* karena merujuk pada jumlah orang.

In [11]:
# to int
hlt01[['resp_admissions_under5', 'resp_admissions_over60', 'asthma_emergency_cases', 'copd_admissions', 'resp',	'cvd', 'death']] = hlt01[['resp_admissions_under5', 'resp_admissions_over60', 'asthma_emergency_cases', 'copd_admissions', 'resp',	'cvd', 'death']].astype('int')
hlt01.dtypes

hlt_id                            object
date                      datetime64[ns]
time                               int64
hospital_id                       object
dow                               object
temp_c                           float64
dptp_c                           float64
rhum_pct                         float64
resp_admissions_under5             int64
resp_admissions_over60             int64
asthma_emergency_cases             int64
copd_admissions                    int64
resp                               int64
cvd                                int64
death                              int64
dtype: object

#### **STEP 5: EXPORT DATA**
Simpan data berupa csv pada folder `[2] Data Processing`

In [12]:
hlt01.to_csv('../[3] CLEAN Dataset DA Gelar Rasa 2026/healt_outcomes_data.csv', sep=',', index=False)
print('done')

done
